<a href="https://colab.research.google.com/github/Ea-mjolnir/Microwave-Remote-sensing/blob/main/Visualization_geojson_io_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
# Import necessary libraries
import json
import folium
from google.colab import files
import io
from IPython.display import clear_output

# Upload the text files
uploaded = files.upload()

# Function to parse GeoJSON from a single file
def parse_geojson_file(file_content):
    file_content = file_content.strip()
    features = []

    # Try parsing as a single JSON object
    try:
        geo_data = json.loads(file_content)
        if isinstance(geo_data, list):
            features = [f for f in geo_data if f.get("type") == "Feature"]
        elif geo_data.get("type") == "FeatureCollection":
            features = geo_data.get("features", [])
        elif geo_data.get("type") == "Feature":
            features = [geo_data]
        elif "geometry" in geo_data:
            features = [{
                "type": "Feature",
                "geometry": geo_data.get("geometry", {}),
                "properties": geo_data.get("properties", {})
            }]
        else:
            print(f"Invalid GeoJSON type: {geo_data.get('type')}")
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON as single object: {e}")
        # Split into multiple JSON objects
        objects = []
        current = ""
        brace_count = 0
        for char in file_content:
            current += char
            if char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1
                if brace_count == 0:
                    objects.append(current.strip())
                    current = ""
        # Skip trailing invalid data (e.g., ])
        if current.strip() and not current.strip().startswith('{'):
            print(f"Ignoring trailing invalid data: {current.strip()[:50]}...")

        # Parse each object
        for i, obj in enumerate(objects):
            try:
                parsed = json.loads(obj)
                if parsed.get("type") == "Feature":
                    features.append(parsed)
                elif parsed.get("type") == "FeatureCollection":
                    features.extend(parsed.get("features", []))
                elif "geometry" in parsed:
                    features.append({
                        "type": "Feature",
                        "geometry": parsed.get("geometry", {}),
                        "properties": parsed.get("properties", {})
                    })
                else:
                    print(f"Skipping object {i + 1}: Invalid GeoJSON type")
            except json.JSONDecodeError as e2:
                print(f"Skipping object {i + 1}: Invalid JSON - {e2}")
                continue

    return features

# Function to visualize all GeoJSON data
def visualize_all_geojson(uploaded_files):
    all_features = []

    # Process each uploaded file
    for filename in uploaded_files.keys():
        print(f"\nProcessing file: {filename}")
        content = uploaded_files[filename].decode('utf-8')
        features = parse_geojson_file(content)
        print(f"Found {len(features)} features in {filename}")
        all_features.extend(features)

    # Validate features
    if not all_features:
        print("No valid Features found in any file.")
        return None

    # Create FeatureCollection
    geo_data = {
        "type": "FeatureCollection",
        "features": all_features
    }

    # Get the first coordinate for map centering
    try:
        first_feature = geo_data["features"][0]
        first_coord = first_feature["geometry"]["coordinates"][0][0]
    except (IndexError, KeyError) as e:
        print(f"Error accessing coordinates: {e}")
        return None

    # Create a map centered around the first coordinate with OpenStreetMap tiles
    m = folium.Map(
        location=[first_coord[1], first_coord[0]],
        zoom_start=15,
        tiles="OpenStreetMap"
    )

    # Add the GeoJSON to the map with explicit green styling
    folium.GeoJson(
        geo_data,
        style_function=lambda feature: {
            "fillColor": "#00FF00",
            "color": "#00FF00",
            "weight": 2,
            "fillOpacity": 0.5,
            "opacity": 1.0
        },
        tooltip="Polygon"
    ).add_to(m)

    # Print GeoJSON for debugging
    print("\nFinal GeoJSON Data:", json.dumps(geo_data, indent=2))
    print(f"Total number of polygons rendered: {len(geo_data['features'])}")

    return m

# Process and display all uploaded files
clear_output(wait=True)  # Clear output once before displaying the final map
map_object = visualize_all_geojson(uploaded)
if map_object:
    display(map_object)
else:
    print("Failed to generate map. Please check the input files.")


Processing file: farm_16.txt
Error parsing JSON as single object: Extra data: line 272 column 3 (char 6627)
Ignoring trailing invalid data: ]
}...
Found 1 features in farm_16.txt

Processing file: farm_15.txt
Error parsing JSON as single object: Extra data: line 272 column 3 (char 6628)
Ignoring trailing invalid data: ]
}...
Found 1 features in farm_15.txt

Processing file: farm_14.txt
Error parsing JSON as single object: Extra data: line 272 column 3 (char 6617)
Ignoring trailing invalid data: ]
}...
Found 1 features in farm_14.txt

Processing file: farm_13.txt
Error parsing JSON as single object: Extra data: line 272 column 3 (char 6638)
Ignoring trailing invalid data: ]
}...
Found 1 features in farm_13.txt

Processing file: farm_12.txt
Error parsing JSON as single object: Extra data: line 272 column 3 (char 6642)
Ignoring trailing invalid data: ]
}...
Found 1 features in farm_12.txt

Processing file: farm_11.txt
Error parsing JSON as single object: Extra data: line 272 column 3 (ch